In [1]:
#2021.10.20. TUE
#Team_SmokeFree

## SCORE CALCULATION FOR MCLP
#00. 패키지 호출하기. 
import warnings
import pandas as pd 
import geopandas as gpd 
import numpy as np 
from fiona.crs import from_epsg
from pulp import *

#00-1. warning message ignore 
warnings.filterwarnings(action='ignore')


In [2]:
#01. 데이터셋 전처리하기. 
#(1) RANDOM_POINT 데이터셋 불러오기. 
RANDOM_POINT_G = pd.read_excel('../data/RANDOM_POINT_pp.xlsx', sheet_name='금천구_Filter')

#(2) geometry 추가하기. 
RANDOM_POINT_G_point = gpd.points_from_xy(RANDOM_POINT_G['경도'], RANDOM_POINT_G['위도'])

#(3) 데이터셋 재구축하기. 
RANDOM_POINT_G = gpd.GeoDataFrame(RANDOM_POINT_G[RANDOM_POINT_G.columns], geometry=RANDOM_POINT_G_point, crs=from_epsg(4326))

#(4) 좌표계 변환하기. 
RANDOM_POINT_G = RANDOM_POINT_G.to_crs(epsg=5181)

#(5) 처리 결과 확인하기. 
RANDOM_POINT_G

,rand_point,자치구,행정동_코드,행정동,세대_COUNT,노래연습장_COUNT,당구장_COUNT,PC방_COUNT,금연구역_10m_COUNT,금연구역_50m_COUNT,...,음식점_회식_COUNT,음식점_카페_COUNT,SCORE_b,SCORE_b_GARA,SCORE_s,SCORE_sum,SCORE_sum_GARA,위도,경도,geometry
0,30237,금천구,1154551000,가산동,0.695848,7,3,0,2,0,...,24,1,15490.810621,7745.405311,1417,8453.905311,4581.202655,37.481669,126.883167,POINT (189666.433 442475.999)
1,30062,금천구,1154551000,가산동,0.695848,1,3,0,3,0,...,8,2,15574.790621,7787.395311,1374,8474.395311,4580.697655,37.480007,126.883725,POINT (189715.563 442291.533)
2,30241,금천구,1154551000,가산동,0.695848,7,2,0,2,0,...,23,1,15243.910621,7621.955311,1391,8317.455311,4506.477655,37.481549,126.883127,POINT (189662.896 442462.677)
3,30100,금천구,1154551000,가산동,0.695848,2,3,0,4,0,...,12,2,14721.590621,7360.795311,1307,8014.295311,4333.897655,37.480857,126.884488,POINT (189783.219 442385.786)
4,30110,금천구,1154551000,가산동,0.695848,2,3,0,4,0,...,12,2,14721.590621,7360.795311,1307,8014.295311,4333.897655,37.480910,126.884618,POINT (189794.700 442391.591)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7774,33107,금천구,1154570000,시흥제4동,0.297406,0,0,0,1,0,...,7,0,-201.147062,-100.573531,15,-93.073531,-42.786766,37.461580,126.904626,POINT (191562.193 440244.260)
7775,33110,금천구,1154570000,시흥제4동,0.297406,0,0,0,1,0,...,7,0,-201.147062,-100.573531,15,-93.073531,-42.786766,37.461657,126.904704,POINT (191569.081 440252.827)
7776,27204,금천구,1154567000,시흥제1동,0.673139,0,0,0,1,0,...,7,0,-105.556667,-52.778333,-57,-81.278333,-54.889167,37.461627,126.904551,POINT (191555.589 440249.471)
7777,27088,금천구,1154567000,시흥제1동,0.673139,0,0,0,0,0,...,7,0,-144.446667,-72.223333,-58,-101.223333,-65.111667,37.461552,126.904547,POINT (191555.161 440241.139)


In [3]:
#(6) BUFFER 데이터셋 만들기. 
RANDOM_POINT_G_BUFFER = RANDOM_POINT_G.copy()

#(7) BUFFER 지정하기. 
RANDOM_POINT_G_BUFFER['geometry'] = RANDOM_POINT_G_BUFFER.buffer(200)

#(8) 처리 결과 확인하기. 
RANDOM_POINT_G_BUFFER.head()

,rand_point,자치구,행정동_코드,행정동,세대_COUNT,노래연습장_COUNT,당구장_COUNT,PC방_COUNT,금연구역_10m_COUNT,금연구역_50m_COUNT,...,음식점_회식_COUNT,음식점_카페_COUNT,SCORE_b,SCORE_b_GARA,SCORE_s,SCORE_sum,SCORE_sum_GARA,위도,경도,geometry
0,30237,금천구,1154551000,가산동,0.695848,7,3,0,2,0,...,24,1,15490.810621,7745.405311,1417,8453.905311,4581.202655,37.481669,126.883167,"POLYGON ((189866.433 442475.999, 189865.469 44..."
1,30062,금천구,1154551000,가산동,0.695848,1,3,0,3,0,...,8,2,15574.790621,7787.395311,1374,8474.395311,4580.697655,37.480007,126.883725,"POLYGON ((189915.563 442291.533, 189914.600 44..."
2,30241,금천구,1154551000,가산동,0.695848,7,2,0,2,0,...,23,1,15243.910621,7621.955311,1391,8317.455311,4506.477655,37.481549,126.883127,"POLYGON ((189862.896 442462.677, 189861.933 44..."
3,30100,금천구,1154551000,가산동,0.695848,2,3,0,4,0,...,12,2,14721.590621,7360.795311,1307,8014.295311,4333.897655,37.480857,126.884488,"POLYGON ((189983.219 442385.786, 189982.256 44..."
4,30110,금천구,1154551000,가산동,0.695848,2,3,0,4,0,...,12,2,14721.590621,7360.795311,1307,8014.295311,4333.897655,37.480910,126.884618,"POLYGON ((189994.700 442391.591, 189993.737 44..."


In [4]:
#02. MCLP 알고리즘 지정하기. 
#(1) PROCESS 01
NUM = len(RANDOM_POINT_G)
final = []
for i in range(NUM) :
    temp = pd.DataFrame()
    rest = RANDOM_POINT_G.copy()
    target = gpd.GeoDataFrame(RANDOM_POINT_G_BUFFER.iloc[i,:]).transpose()
    target.crs = "EPSG:5181"
    for j in range(10):
        temp = pd.concat([temp,RANDOM_POINT_G[RANDOM_POINT_G.index.isin(target['rand_point'].index)]], axis=0)
        inp, res = rest.sindex.query_bulk(target.geometry, predicate='intersects') 
        rest = rest.drop(rest.iloc[res,:].index)
        target = gpd.GeoDataFrame(RANDOM_POINT_G_BUFFER.iloc[rest.loc[:, 'SCORE_sum_GARA'].idxmax(),:]).transpose()
        target.crs = "EPSG:5181"
    final.append(temp['SCORE_sum_GARA'].sum())
    print(f'DONE ! INDEX_NUM = {i}')

DONE ! INDEX_NUM = 0
DONE ! INDEX_NUM = 1
DONE ! INDEX_NUM = 2
DONE ! INDEX_NUM = 3
DONE ! INDEX_NUM = 4
DONE ! INDEX_NUM = 5
DONE ! INDEX_NUM = 6
DONE ! INDEX_NUM = 7
DONE ! INDEX_NUM = 8
DONE ! INDEX_NUM = 9
DONE ! INDEX_NUM = 10
DONE ! INDEX_NUM = 11
DONE ! INDEX_NUM = 12
DONE ! INDEX_NUM = 13
DONE ! INDEX_NUM = 14
DONE ! INDEX_NUM = 15
DONE ! INDEX_NUM = 16
DONE ! INDEX_NUM = 17
DONE ! INDEX_NUM = 18
DONE ! INDEX_NUM = 19
DONE ! INDEX_NUM = 20
DONE ! INDEX_NUM = 21
DONE ! INDEX_NUM = 22
DONE ! INDEX_NUM = 23
DONE ! INDEX_NUM = 24
DONE ! INDEX_NUM = 25
DONE ! INDEX_NUM = 26
DONE ! INDEX_NUM = 27
DONE ! INDEX_NUM = 28
DONE ! INDEX_NUM = 29
DONE ! INDEX_NUM = 30
DONE ! INDEX_NUM = 31
DONE ! INDEX_NUM = 32
DONE ! INDEX_NUM = 33
DONE ! INDEX_NUM = 34
DONE ! INDEX_NUM = 35
DONE ! INDEX_NUM = 36
DONE ! INDEX_NUM = 37
DONE ! INDEX_NUM = 38
DONE ! INDEX_NUM = 39
DONE ! INDEX_NUM = 40
DONE ! INDEX_NUM = 41
DONE ! INDEX_NUM = 42
DONE ! INDEX_NUM = 43
DONE ! INDEX_NUM = 44
DONE ! INDEX_NUM = 4

In [5]:
#(2) PROCESS_02
RANDOM_POINT_G_FINAL_LIST = pd.DataFrame()
rest = RANDOM_POINT_G.copy()
target = gpd.GeoDataFrame(RANDOM_POINT_G_BUFFER.iloc[final.index(max(final)),:]).transpose()
target.crs = "EPSG:5181"
for j in range(10):
    RANDOM_POINT_G_FINAL_LIST = pd.concat([RANDOM_POINT_G_FINAL_LIST,RANDOM_POINT_G[RANDOM_POINT_G.index.isin(target['rand_point'].index)]],axis=0)
    inp, res = rest.sindex.query_bulk(target.geometry, predicate='intersects') 
    rest = rest.drop(rest.iloc[res,:].index)
    target = gpd.GeoDataFrame(RANDOM_POINT_G_BUFFER.iloc[rest.loc[:,'SCORE_sum_GARA'].idxmax(),:]).transpose()
    target.crs = "EPSG:5181"
RANDOM_POINT_G_FINAL_LIST = RANDOM_POINT_G_FINAL_LIST.to_crs(5181)

In [6]:
#(3) PROCESS 03
RANDOM_POINT_G_FINAL_LIST               = pd.DataFrame(RANDOM_POINT_G_FINAL_LIST.sort_values(by='SCORE_sum_GARA',ascending=False).reset_index(drop=True))
RANDOM_POINT_G_FINAL_LIST['RANK']       = RANDOM_POINT_G_FINAL_LIST.index+1
RANDOM_POINT_G_FINAL_LIST['rand_point'] = RANDOM_POINT_G_FINAL_LIST['rand_point'].astype(str)

#(4) 처리 결과 확인하기. 
RANDOM_POINT_G_FINAL_LIST

,rand_point,자치구,행정동_코드,행정동,세대_COUNT,노래연습장_COUNT,당구장_COUNT,PC방_COUNT,금연구역_10m_COUNT,금연구역_50m_COUNT,...,음식점_카페_COUNT,SCORE_b,SCORE_b_GARA,SCORE_s,SCORE_sum,SCORE_sum_GARA,위도,경도,geometry,RANK
0,30237,금천구,1154551000,가산동,0.695848,7,3,0,2,0,...,1,15490.810621,7745.405311,1417,8453.905311,4581.202655,37.481669,126.883167,POINT (189666.433 442475.999),1
1,26288,금천구,1154567000,시흥제1동,0.673139,10,1,1,0,0,...,0,13583.843333,6791.921667,-379,6602.421667,3206.460833,37.454380,126.902257,POINT (191351.793 439445.445),2
2,29986,금천구,1154551000,가산동,0.695848,1,1,0,3,0,...,1,10687.640621,5343.820311,942,5814.820311,3142.910155,37.480002,126.881629,POINT (189530.161 442291.160),3
3,30296,금천구,1154551000,가산동,0.695848,1,1,0,0,0,...,1,10087.510621,5043.755311,892,5489.755311,2967.877655,37.482099,126.880914,POINT (189467.254 442524.014),4
4,29662,금천구,1154551000,가산동,0.695848,15,2,5,2,0,...,2,9829.690621,4914.845311,856,5342.845311,2885.422655,37.477863,126.891128,POINT (190370.097 442052.827),5
5,29490,금천구,1154551000,가산동,0.695848,0,0,0,3,0,...,0,9762.680621,4881.340311,839,5300.840311,2860.170155,37.477734,126.888118,POINT (190103.884 442038.764),6
6,29537,금천구,1154551000,가산동,0.695848,0,2,0,2,0,...,1,9370.440621,4685.220311,794,5082.220311,2739.610155,37.476980,126.883008,POINT (189651.717 441955.658),7
7,28081,금천구,1154561000,독산제1동,1.000000,1,1,0,3,0,...,0,11218.910000,5609.455000,-142,5538.455000,2733.727500,37.469757,126.896051,POINT (190804.598 441152.673),8
8,29760,금천구,1154551000,가산동,0.695848,0,1,0,1,0,...,0,7553.280621,3776.640311,650,4101.640311,2213.320155,37.478668,126.884314,POINT (189767.531 442142.832),9
9,25989,금천구,1154567000,시흥제1동,0.673139,5,2,3,0,0,...,0,9235.793333,4617.896667,-305,4465.396667,2156.448333,37.451881,126.901864,POINT (191316.683 439168.073),10


In [7]:
#(5) 처리 결과 저장하기.
RANDOM_POINT_G_FINAL_LIST.to_excel('RANDOM_POINT_G_RANK.xlsx', index=False)